# Gold To SQL Ingestion Using SQL Alchemy

- Load up imports

In [1]:
from sqlalchemy import create_engine, text, Integer, Float, Date, String, BigInteger
import urllib

import pandas as pd


### Load Up Dataframes for each table

In [2]:
dimCargoMeasurementCSV = pd.read_csv('GoldLayerData/dim_cargo_measurement.csv')
dimConsigneeCSV = pd.read_csv('GoldLayerData/dim_consignee.csv')
dimPortCSV = pd.read_csv('GoldLayerData/dim_port.csv')
dimShipperCSV = pd.read_csv('GoldLayerData/dim_shipper.csv')
dimVesselCSV = pd.read_csv('GoldLayerData/dim_vessel.csv')
factHeaderCSV = pd.read_csv('GoldLayerData/fact_header.csv')

In [3]:
dimCargoMeasurementCSV

,identifier,manifest_unit,measurement_unit
0,201901012,PKG,Cubic Meters
1,201901013,PKG,Cubic Meters
2,201901014,CTN,Cubic Meters
3,201901015,PCL,Cubic Meters
4,201901016,CTN,Cubic Meters
...,...,...,...
34038974,2020092382684,CTN,NaN
34038975,2020092382685,CTN,NaN
34038976,2020092382686,CTN,NaN
34038977,2020092382687,PKG,Cubic Meters


In [4]:
dimConsigneeCSV.dtypes

consignee_id           int64
consignee_name           str
consignee_address_1      str
consignee_address_2      str
consignee_address_3      str
dtype: object

In [5]:
dimPortCSV.dtypes

port_id      int64
port_name      str
dtype: object

In [6]:
dimShipperCSV.dtypes

shipper_id                 int64
shipper_party_name           str
shipper_party_address_1      str
shipper_party_address_2      str
shipper_party_address_3      str
dtype: object

In [7]:
dimVesselCSV

,vessel_id,vessel_country_code,vessel_name
0,1,AD,ACACIA MING
1,2,AD,AGAMEMNON
2,3,AD,AIN SNAN
3,4,AD,AL QIBLA
4,5,AD,ALLEGRO
...,...,...,...
38093,38094,NaN,ONE CRANE
38094,38095,NaN,ONE CYGNUS
38095,38096,NaN,SCHUBERT
38096,38097,NaN,YM ENLIGHTENMENT


In [8]:
factHeaderCSV

,identifier,estimated_arrival_date,actual_arrival_date,port_of_unlading_id,foreign_port_of_lading_qualifier_id,foreign_port_of_lading_id,port_of_destination_id,foreign_port_of_destination_qualifier_id,foreign_port_of_destination_id,vessel_id,shipper_id,consignee_id,manifest_quantity,weight,measurement
0,201901012,2018-03-28,2018-03-30,887,1269,1176,NaN,NaN,NaN,27388,NaN,NaN,3350,44599,12756
1,201901013,2018-03-28,2018-03-30,887,1269,207,NaN,NaN,NaN,27388,NaN,NaN,642,9258,2539
2,201901014,2018-03-28,2018-03-30,887,1269,207,NaN,NaN,NaN,27388,NaN,NaN,2743,34760,136
3,201901015,2018-03-28,2018-03-30,887,1269,207,NaN,NaN,NaN,27388,1296312.0,1559462.0,37,19537,0
4,201901016,2018-03-28,2018-03-30,887,1269,207,NaN,NaN,NaN,27388,NaN,NaN,3469,42107,6003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34038974,2020092382684,2020-09-08,2020-09-11,1273,1269,1629,NaN,NaN,NaN,37472,675504.0,1997811.0,386,7370,0
34038975,2020092382685,2020-07-19,2020-07-20,712,1269,1288,NaN,NaN,NaN,27003,3294336.0,409782.0,338,9633,0
34038976,2020092382686,2020-07-19,2020-07-20,712,1269,1288,NaN,NaN,NaN,27003,3294336.0,409782.0,432,7906,0
34038977,2020092382687,2020-07-20,2020-07-20,716,1269,1623,NaN,NaN,NaN,32116,NaN,NaN,40,19750,99


### Windows Authentication to SQL Database

- Build the SQLAlchemy Connection String

In [9]:
# Build the SQLAlchemy Connection String
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=CargoImporters;"
    "Trusted_Connection=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}", fast_executemany=True)


- Test the Connection

In [10]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print(result.fetchone())

(1,)


---

In [11]:
from math import ceil

def load_with_progress(
    df,
    table_name,
    engine,
    schema="dbo",
    chunksize=25_000,
    dtype=None
):
    total_rows = len(df)
    total_chunks = ceil(total_rows / chunksize)

    for i, start in enumerate(range(0, total_rows, chunksize), start=1):
        end = min(start + chunksize, total_rows)
        chunk = df.iloc[start:end]

        chunk.to_sql(
            table_name,
            engine,
            schema=schema,
            if_exists="replace" if i == 1 else "append",
            index=False,
            method=None,
            dtype=dtype
        )

        print(
            f"✅ Loaded rows {start:,}–{end:,} "
            f"({end:,}/{total_rows:,}) | "
            f"Chunk {i}/{total_chunks}"
        )


### Load Up Data to SQL Database

- dim_cargo_measurement table

In [12]:
load_with_progress(
    df=dimCargoMeasurementCSV,
    table_name="dim_cargo_measurement",
    engine=engine,
    schema="dbo",
    chunksize=25_000,
    dtype={
        "identifier": BigInteger(),
        "manifest_unit": String(10),
        "measurement_unit": String(50)
    }
)

✅ Loaded rows 0–25,000 (25,000/34,038,979) | Chunk 1/1362
✅ Loaded rows 25,000–50,000 (50,000/34,038,979) | Chunk 2/1362
✅ Loaded rows 50,000–75,000 (75,000/34,038,979) | Chunk 3/1362
✅ Loaded rows 75,000–100,000 (100,000/34,038,979) | Chunk 4/1362
✅ Loaded rows 100,000–125,000 (125,000/34,038,979) | Chunk 5/1362
✅ Loaded rows 125,000–150,000 (150,000/34,038,979) | Chunk 6/1362
✅ Loaded rows 150,000–175,000 (175,000/34,038,979) | Chunk 7/1362
✅ Loaded rows 175,000–200,000 (200,000/34,038,979) | Chunk 8/1362
✅ Loaded rows 200,000–225,000 (225,000/34,038,979) | Chunk 9/1362
✅ Loaded rows 225,000–250,000 (250,000/34,038,979) | Chunk 10/1362
✅ Loaded rows 250,000–275,000 (275,000/34,038,979) | Chunk 11/1362
✅ Loaded rows 275,000–300,000 (300,000/34,038,979) | Chunk 12/1362
✅ Loaded rows 300,000–325,000 (325,000/34,038,979) | Chunk 13/1362
✅ Loaded rows 325,000–350,000 (350,000/34,038,979) | Chunk 14/1362
✅ Loaded rows 350,000–375,000 (375,000/34,038,979) | Chunk 15/1362
✅ Loaded rows 375,0

- dim_consignee table

In [13]:
load_with_progress(
    df=dimConsigneeCSV,
    table_name="dim_consignee",
    engine=engine,
    schema="dbo",
    chunksize=25_000,
    dtype={
        "consignee_id": BigInteger(),
        "consignee_name": String(255),
        "consignee_address_1": String(255),
        "consignee_address_2": String(255),
        "Consignee_address_3": String(255)
    }
)

✅ Loaded rows 0–25,000 (25,000/3,767,444) | Chunk 1/151
✅ Loaded rows 25,000–50,000 (50,000/3,767,444) | Chunk 2/151
✅ Loaded rows 50,000–75,000 (75,000/3,767,444) | Chunk 3/151
✅ Loaded rows 75,000–100,000 (100,000/3,767,444) | Chunk 4/151
✅ Loaded rows 100,000–125,000 (125,000/3,767,444) | Chunk 5/151
✅ Loaded rows 125,000–150,000 (150,000/3,767,444) | Chunk 6/151
✅ Loaded rows 150,000–175,000 (175,000/3,767,444) | Chunk 7/151
✅ Loaded rows 175,000–200,000 (200,000/3,767,444) | Chunk 8/151
✅ Loaded rows 200,000–225,000 (225,000/3,767,444) | Chunk 9/151
✅ Loaded rows 225,000–250,000 (250,000/3,767,444) | Chunk 10/151
✅ Loaded rows 250,000–275,000 (275,000/3,767,444) | Chunk 11/151
✅ Loaded rows 275,000–300,000 (300,000/3,767,444) | Chunk 12/151
✅ Loaded rows 300,000–325,000 (325,000/3,767,444) | Chunk 13/151
✅ Loaded rows 325,000–350,000 (350,000/3,767,444) | Chunk 14/151
✅ Loaded rows 350,000–375,000 (375,000/3,767,444) | Chunk 15/151
✅ Loaded rows 375,000–400,000 (400,000/3,767,444)

- dim_port table

In [14]:
load_with_progress(
    df=dimPortCSV,
    table_name="dim_port",
    engine=engine,
    schema="dbo",
    chunksize=25_000,
    dtype={
        "port_id": BigInteger(),
        "port_name": String(255),
    }
)

✅ Loaded rows 0–1,665 (1,665/1,665) | Chunk 1/1


- dim_shipper table

In [15]:
load_with_progress(
    df=dimShipperCSV,
    table_name="dim_shipper",
    engine=engine,
    schema="dbo",
    chunksize=25_000,
    dtype={
        "shipper_id": BigInteger(),
        "shipper_party_name": String(255),
        "consignee_address_1": String(255),
        "consignee_address_2": String(255),
        "Consignee_address_3": String(255)
    }
)

✅ Loaded rows 0–25,000 (25,000/3,371,079) | Chunk 1/135
✅ Loaded rows 25,000–50,000 (50,000/3,371,079) | Chunk 2/135
✅ Loaded rows 50,000–75,000 (75,000/3,371,079) | Chunk 3/135
✅ Loaded rows 75,000–100,000 (100,000/3,371,079) | Chunk 4/135
✅ Loaded rows 100,000–125,000 (125,000/3,371,079) | Chunk 5/135
✅ Loaded rows 125,000–150,000 (150,000/3,371,079) | Chunk 6/135
✅ Loaded rows 150,000–175,000 (175,000/3,371,079) | Chunk 7/135
✅ Loaded rows 175,000–200,000 (200,000/3,371,079) | Chunk 8/135
✅ Loaded rows 200,000–225,000 (225,000/3,371,079) | Chunk 9/135
✅ Loaded rows 225,000–250,000 (250,000/3,371,079) | Chunk 10/135
✅ Loaded rows 250,000–275,000 (275,000/3,371,079) | Chunk 11/135
✅ Loaded rows 275,000–300,000 (300,000/3,371,079) | Chunk 12/135
✅ Loaded rows 300,000–325,000 (325,000/3,371,079) | Chunk 13/135
✅ Loaded rows 325,000–350,000 (350,000/3,371,079) | Chunk 14/135
✅ Loaded rows 350,000–375,000 (375,000/3,371,079) | Chunk 15/135
✅ Loaded rows 375,000–400,000 (400,000/3,371,079)

- dim_vessel table

In [16]:
load_with_progress(
    df=dimVesselCSV,
    table_name="dim_vessel",
    engine=engine,
    schema="dbo",
    chunksize=25_000,
    dtype={
        "vessel_id": BigInteger(),
        "vessel_country_code": String(30),
        "consignee_name": String(255),
    }
)

✅ Loaded rows 0–25,000 (25,000/38,098) | Chunk 1/2
✅ Loaded rows 25,000–38,098 (38,098/38,098) | Chunk 2/2


- fact_header table

In [17]:
load_with_progress(
    df=factHeaderCSV,
    table_name="fact_header",
    engine=engine,
    schema="dbo",
    chunksize=25_000,
    dtype={
        "identifier": BigInteger(),
        "estimated_arrival_date": Date(),
        "actual_arrival_date": Date(),
        "port_of_unlading_id": BigInteger(),
        "foreign_port_of_lading_qualifier_id": BigInteger(),
        "foreign_port_of_lading_id": BigInteger(),
        "port_of_destination_id": BigInteger(),
        "foreign_port_of_destination_qualifier_id": BigInteger(),
        "foreign_port_of_destination_id": BigInteger(),
        "vessel_id": BigInteger(),
        "shipper_id": BigInteger(),
        "consignee_id": BigInteger(),
        "manifest_quantity": BigInteger(),
        "weight": BigInteger(),
        "measurement": BigInteger()
    }
)

✅ Loaded rows 0–25,000 (25,000/34,038,979) | Chunk 1/1362
✅ Loaded rows 25,000–50,000 (50,000/34,038,979) | Chunk 2/1362
✅ Loaded rows 50,000–75,000 (75,000/34,038,979) | Chunk 3/1362
✅ Loaded rows 75,000–100,000 (100,000/34,038,979) | Chunk 4/1362
✅ Loaded rows 100,000–125,000 (125,000/34,038,979) | Chunk 5/1362
✅ Loaded rows 125,000–150,000 (150,000/34,038,979) | Chunk 6/1362
✅ Loaded rows 150,000–175,000 (175,000/34,038,979) | Chunk 7/1362
✅ Loaded rows 175,000–200,000 (200,000/34,038,979) | Chunk 8/1362
✅ Loaded rows 200,000–225,000 (225,000/34,038,979) | Chunk 9/1362
✅ Loaded rows 225,000–250,000 (250,000/34,038,979) | Chunk 10/1362
✅ Loaded rows 250,000–275,000 (275,000/34,038,979) | Chunk 11/1362
✅ Loaded rows 275,000–300,000 (300,000/34,038,979) | Chunk 12/1362
✅ Loaded rows 300,000–325,000 (325,000/34,038,979) | Chunk 13/1362
✅ Loaded rows 325,000–350,000 (350,000/34,038,979) | Chunk 14/1362
✅ Loaded rows 350,000–375,000 (375,000/34,038,979) | Chunk 15/1362
✅ Loaded rows 375,0